<!-- Cell 1 -->
# BolChehra — Talking Face Generator (CLEAN, FINAL)
Saare fixes consolidated hain. Order mein, upar se neeche run karo — koi cell skip mat karo (jab tak 'OPTIONAL' na likha ho).

In [ ]:
# Cell 2
# STEP 0 - GPU check
!nvidia-smi


In [ ]:
# Cell 3
# STEP 1 - Python 3.8 setup (SadTalker ke liye)
!apt-get update -qq
!apt-get install -y -qq software-properties-common
!add-apt-repository -y ppa:deadsnakes/ppa
!apt-get update -qq
!apt-get install -y -qq python3.8 python3.8-distutils python3.8-dev python3.8-venv build-essential
!curl -sS https://bootstrap.pypa.io/pip/3.8/get-pip.py -o get-pip.py
!python3.8 get-pip.py
!python3.8 --version
!python3.8 -m pip --version

In [ ]:
# Cell 4
# STEP 2 - SadTalker clone + torch (GPU version)
%cd /content
!git clone https://github.com/Winfredy/SadTalker.git
%cd /content/SadTalker

!python3.8 -m pip install torch==1.13.1+cu117 torchvision==0.14.1+cu117 torchaudio==0.13.1 \
    --extra-index-url https://download.pytorch.org/whl/cu117
!pwd


In [ ]:
# Cell 5
# STEP 3 - SadTalker dependencies
!python3.8 -m pip install --no-input -q opencv-python numpy==1.23.4
!python3.8 -m pip install --no-input -q lmdb==1.4.1 --only-binary=:all:
!python3.8 -m pip install --no-input -q basicsr --no-deps
!python3.8 -m pip install --no-input -q facexlib --no-deps
!python3.8 -m pip install --no-input -q gfpgan --no-deps
!python3.8 -m pip install --no-input -q addict future yapf tb-nightly numba tqdm PyYAML yacs kornia safetensors scikit-image filterpy
!python3.8 -m pip install --no-input -q pydub
!python3.8 -m pip install --no-input -q librosa==0.9.2 numba resampy
!python3.8 -m pip install --no-input -q imageio==2.19.3 imageio-ffmpeg==0.4.7 --force-reinstall --no-deps
!python3.8 -m pip install --no-input -q gTTS

!python3.8 -c "import cv2, facexlib, basicsr, gfpgan; print('All core modules OK')"


In [ ]:
# Cell 6
# STEP 4 - Pre-trained models download
%cd /content/SadTalker
!rm -rf checkpoints
!bash scripts/download_models.sh
!ls -la checkpoints/


<!-- Cell 7 -->
## Ngrok token setup (SECURE way — kabhi bhi token seedha code mein mat likho)
1. Left sidebar mein 🔑 (key icon / "Secrets") pe click karo
2. "Add new secret" — Name: `NGROK_TOKEN`, Value: apna token (dashboard.ngrok.com se)
3. Toggle "Notebook access" ON karo


<!-- Cell 8 -->
## Voice Cloning Setup (XTTS-v2) — ALL FIXES CONSOLIDATED
Ye ek hi cell mein saare version-fixes hain jo trial-and-error se pata chale:
- `setuptools<70` (warna `pkg_resources` missing error)
- `transformers==4.36.2` (naya version `BeamSearchScorer` import todta hai)
- `torch==2.5.1` + `torchaudio==2.5.1` (matching versions, warna checkpoint load fail hota hai)
- `COQUI_TOS_AGREED=1` (warna interactive TOS prompt subprocess mein crash karta hai)


In [ ]:
# Cell 9
# STEP 5 - XTTS-v2 voice cloning environment (Python 3.10, SadTalker se isolated)
!apt-get install -y -qq python3.10 python3.10-venv python3.10-dev

!python3.10 -m venv /content/xtts_env
!/content/xtts_env/bin/pip install --upgrade pip -q
!/content/xtts_env/bin/pip install TTS -q
!/content/xtts_env/bin/pip install "transformers==4.36.2" -q
!/content/xtts_env/bin/pip install "torch==2.5.1" "torchaudio==2.5.1" -q

# ZAROORI: setuptools ko SABSE LAST mein pin karo, taaki upar wali installs isko overwrite na karein
!/content/xtts_env/bin/pip install "setuptools<70" -q

xtts_download_script = '''
from TTS.api import TTS
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2", progress_bar=True).to("cuda")
print("XTTS model ready")
'''
with open("/content/_xtts_download.py", "w") as f:
    f.write(xtts_download_script)

!COQUI_TOS_AGREED=1 /content/xtts_env/bin/python /content/_xtts_download.py

In [ ]:
# NAYA CELL — isse Cell 10 se PEHLE, ek baar run karo (har fresh Colab session mein).
!/content/xtts_env/bin/pip install pydub -q
print("pydub install ho gayi.")

In [ ]:
# Cell 10
xtts_script = '''
import sys, os
os.environ["COQUI_TOS_AGREED"] = "1"
from TTS.api import TTS
import requests

def transliterate_to_devanagari(text):
    """Roman/Hinglish text ko Devanagari Hindi mein convert karta hai,
    taaki XTTS ka language='hi' sahi se pronounce kar sake."""
    try:
        resp = requests.get(
            "https://inputtools.google.com/request",
            params={"text": text, "itc": "hi-t-i0-und", "num": 1},
            timeout=15
        )
        data = resp.json()
        if data[0] == "SUCCESS":
            words = text.split(" ")
            converted_words = []
            for i, word in enumerate(words):
                suggestions = data[1][i][1] if i < len(data[1]) else None
                converted_words.append(suggestions[0] if suggestions else word)
            return " ".join(converted_words)
    except Exception:
        pass
    return text  # agar API fail ho jaaye, to original text hi use karo

script_text = sys.argv[1]
speaker_wav = sys.argv[2]
output_path = sys.argv[3]

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda")
script_text_hi = transliterate_to_devanagari(script_text)

# ZAROORI: XTTS-v2 ka ek known bug hai — bahut chhoti text (jaise 10-15 words
# se kam) di jaaye to model confuse ho jaata hai ki utterance kahan khatam ho,
# aur audio ko repeat/loop kar deta hai. Do cheezein isse rokti hain:
# 1) text ke end mein punctuation ho (model ko "stop" ka clear signal milta hai)
# 2) chhoti text ke liye sentence-splitting band rakho (split karna hi
#    looping trigger karta hai jab text pehle se hi ek chhota sentence ho)
if not script_text_hi.strip().endswith((".", "?", "!", "।")):
    script_text_hi = script_text_hi.strip() + "."

is_short_text = len(script_text_hi.split()) < 15

tts.tts_to_file(
    text=script_text_hi,
    speaker_wav=speaker_wav,
    language="hi",
    split_sentences=not is_short_text,
    file_path=output_path
)

# ZAROORI: XTTS kabhi-kabhi asli speech bolne ke baad rukta nahi — khud hi
# extra "hallucinated" (bina matlab ki, kisi aur language jaisi lagne wali)
# audio generate karta rehta hai, khaaskar chhoti text ke saath. Iska fix:
# generated audio mein jahan pehli LAMBI khamoshi (pause) milti hai — jo
# asli speech aur hallucinated hisse ke beech hoti hai — wahan se aage sab
# kaat dete hain.
try:
    from pydub import AudioSegment
    from pydub.silence import detect_nonsilent

    audio = AudioSegment.from_file(output_path)
    nonsilent_ranges = detect_nonsilent(
        audio, min_silence_len=600, silence_thresh=audio.dBFS - 16
    )
    if nonsilent_ranges:
        end_ms = nonsilent_ranges[0][1] + 300  # asli speech ke baad thoda buffer
        if end_ms < len(audio):
            trimmed = audio[:end_ms]
            fmt = output_path.split(".")[-1] if "." in output_path else "wav"
            trimmed.export(output_path, format=fmt)
except Exception as trim_err:
    print("Trim step skip hua (audio waise hi rahega):", trim_err)

print("Voice cloned audio ban gaya:", output_path)
'''
with open("/content/xtts_clone.py", "w") as f:
    f.write(xtts_script)
print("xtts_clone.py ban gaya")

<!-- Cell 11 -->
## Background / Foreground / Corner-light Setup (rembg — mediapipe hataya gaya, unreliable tha)

In [ ]:
# Cell 12
# STEP 7 - rembg install (background removal ke liye, stable CPU version)
!pip uninstall onnxruntime-gpu -y -q
!pip install --force-reinstall rembg onnxruntime opencv-python-headless pymatting -q
print('rembg install complete (stable version)')

In [ ]:
# Cell 13
import sys
!{sys.executable} -m pip install --force-reinstall rembg onnxruntime opencv-python-headless pymatting -q
print("Done")

In [ ]:
import onnxruntime
print("Available providers:", onnxruntime.get_available_providers())

In [ ]:
# Cell 14
!pip uninstall numpy -y -q
!pip install numpy -q

<!-- Cell 15 -->
restart and run

In [ ]:
# Cell 16
import numpy as np
print("numpy version:", np.__version__)

from rembg import remove
import cv2
print('rembg + opencv ready')

In [ ]:
# Cell 17
!pip install "numpy<2.0" --force-reinstall -q

In [ ]:
# Cell 18
!pip install "numpy==1.23.4" --force-reinstall -q

In [ ]:
# Cell 19
!nvidia-smi

In [ ]:
# Cell 21
!python3.8 -c "import torch; print(torch.__version__); print(torch.cuda.is_available())"

<!-- Cell 22 -->
cell 19

In [ ]:
# NAYA CELL — isse Cell 23 (FastAPI/video-generation cell) se PEHLE run karo,
# taaki har naye Colab session mein enhancer/GFPGAN ka broadcast bug fix ho jaaye.
#
# Bug: facexlib ki paste_faces_to_input_image() mein normal color images ke liye
# "inv_soft_mask" 2D (H,W) reh jaata hai jabki "pasted_face" 3D (H,W,3) hota hai —
# isi mismatch se "operands could not be broadcast together with shapes (N,N,3) (N,N)"
# error aata hai. Ye patch inv_soft_mask ko hamesha 3D bana deta hai.
#
# NOTE: facexlib is installed in the python3.8 environment used by SadTalker's
# inference.py, NOT in this main Colab kernel — isliye seedha `import facexlib`
# yahan fail hota hai. Hum python3.8 se uska path nikaalte hain, phir file ko
# seedha edit karte hain (isme facexlib import karne ki zaroorat nahi).

import subprocess
import os

result = subprocess.run(
    ["python3.8", "-c", "import facexlib, os; print(os.path.dirname(facexlib.__file__))"],
    capture_output=True, text=True
)

if result.returncode != 0 or not result.stdout.strip():
    print("ERROR: facexlib ka path python3.8 se nahi mila.")
    print(result.stderr[-1000:])
else:
    facexlib_dir = result.stdout.strip()
    target_file = os.path.join(facexlib_dir, "utils", "face_restoration_helper.py")

    with open(target_file, "r") as f:
        content = f.read()

    OLD = "            inv_soft_mask = cv2.GaussianBlur(inv_mask_center, (blur_size + 1, blur_size + 1), 0)\n"
    NEW = (
        "            inv_soft_mask = cv2.GaussianBlur(inv_mask_center, (blur_size + 1, blur_size + 1), 0)\n"
        "            if inv_soft_mask.ndim == 2:\n"
        "                inv_soft_mask = inv_soft_mask[:, :, None]\n"
    )

    if "if inv_soft_mask.ndim == 2:" in content:
        print("facexlib already patched — kuch karne ki zaroorat nahi.")
    elif OLD in content:
        content = content.replace(OLD, NEW, 1)
        with open(target_file, "w") as f:
            f.write(content)
        print("facexlib patch ho gaya:", target_file)
    else:
        print("WARNING: expected line nahi mili — facexlib ka version different ho sakta hai.")
        print("File path:", target_file)

In [ ]:
# NAYA CELL — isse Cell 23 (FastAPI/video-generation cell) se PEHLE run karo.
#
# Pichla patch sirf EK jagah (dist-packages) patch kar raha tha, lekin agar
# system mein iski aur bhi copies hain (duplicate installs), to Python kabhi
# kabhi doosri (unpatched) copy load kar leta hai — isi wajah se error dobara
# aaya. Ye version POORE filesystem mein is file ki har copy dhoondta hai aur
# SABKO patch karta hai, phir verify karta hai ki kaun-kaun patch hui.

import subprocess

OLD = "inv_soft_mask = cv2.GaussianBlur(inv_mask_center, (blur_size + 1, blur_size + 1), 0)"
MARKER = "if inv_soft_mask.ndim == 2:"

# Step 1: poore system mein is file ki saari copies dhoondo
find_result = subprocess.run(
    ["find", "/", "-name", "face_restoration_helper.py", "-not", "-path", "*/node_modules/*"],
    capture_output=True, text=True
)
all_paths = [p for p in find_result.stdout.strip().split("\n") if p.strip()]

if not all_paths:
    print("Koi bhi face_restoration_helper.py file nahi mili — kuch aur galat ho sakta hai.")
else:
    print(f"{len(all_paths)} copy(ies) mili:")
    for p in all_paths:
        print(" -", p)
    print()

    for path in all_paths:
        try:
            with open(path, "r") as f:
                content = f.read()
        except Exception as e:
            print(f"SKIP (read error) {path}: {e}")
            continue

        if MARKER in content:
            print(f"ALREADY PATCHED: {path}")
            continue

        if OLD not in content:
            print(f"WARNING: expected line nahi mili in {path} (shayad different version hai)")
            continue

        # patch: OLD line ke turant baad ndim check add karo, indentation match karte hue
        lines = content.split("\n")
        new_lines = []
        patched = False
        for line in lines:
            new_lines.append(line)
            if OLD in line and not patched:
                indent = line[:len(line) - len(line.lstrip())]
                new_lines.append(f"{indent}if inv_soft_mask.ndim == 2:")
                new_lines.append(f"{indent}    inv_soft_mask = inv_soft_mask[:, :, None]")
                patched = True
        if patched:
            with open(path, "w") as f:
                f.write("\n".join(new_lines))
            print(f"PATCHED: {path}")

    # Step 2: verify — dobara check karo ki sab jagah patch lag gaya
    print("\n--- Verification ---")
    for path in all_paths:
        with open(path, "r") as f:
            ok = MARKER in f.read()
        print(f"{'OK' if ok else 'STILL UNPATCHED'} — {path}")

In [ ]:
# Cell 23
# Ye cell FastAPI + ngrok se ek public URL banata hai jise BolChehra frontend use karega.
import asyncio
import sys
!{sys.executable} -m pip install fastapi nest-asyncio pyngrok uvicorn python-multipart gTTS -q
!pip install "numpy==1.23.4" --force-reinstall -q

import os, threading, re, time
import nest_asyncio, uvicorn, shutil, subprocess, uuid
from pathlib import Path
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, JSONResponse, HTMLResponse
from gtts import gTTS
from pyngrok import ngrok

from google.colab import userdata
ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))

import cv2
import numpy as np
from rembg import remove, new_session
from PIL import Image
rembg_session = new_session("u2net_human_seg")


def center_crop_square(image_path):
    """Photo ko square banata hai — CROP karke nahi, PAD karke, taaki original
    photo ka koi hissa kabhi kaata na jaaye (pehle non-square photos ke top/
    bottom ya left/right se content cut ho raha tha, jisse video "zoomed/
    cropped" dikhti thi). Ab agar photo already square nahi hai, to us par
    zaroorat ke mutabik letterbox padding add hoti hai — asli image bilkul
    as-it-is (full) rehti hai, sirf square banane ke liye extra jagah judti hai.

    Ye square shape browser preview (jo object-fit:cover se dikhta hai) aur
    SadTalker ke actual video frame ka aspect ratio match karne ke liye zaroori
    hai, taaki foreground/mic ki drag-set ki hui position shift na ho."""
    img = Image.open(image_path).convert("RGB")
    w, h = img.size
    side = max(w, h)

    # Padding color: image ke corner pixel ka average — taaki bars jitna ho
    # sake utna background jaisa hi blend ho, ekdum kaale bars na dikhein.
    corner_pixels = [img.getpixel((0, 0)), img.getpixel((w - 1, 0)),
                     img.getpixel((0, h - 1)), img.getpixel((w - 1, h - 1))]
    pad_color = tuple(int(sum(c[i] for c in corner_pixels) / 4) for i in range(3))

    padded = Image.new("RGB", (side, side), pad_color)
    paste_x = (side - w) // 2
    paste_y = (side - h) // 2
    padded.paste(img, (paste_x, paste_y))

    # ZAROORI: bade/odd-dimension source images (jaise 1472x1472) ke saath
    # SadTalker ka "--preprocess full" + "--enhancer" combo internally ek
    # known bug trigger karta hai (paste-back step mein mask shape mismatch —
    # "operands could not be broadcast together" error). Isse bachne ke liye
    # image ko ek fixed, predictable size pe resize kar dete hain (SadTalker
    # ke "--size 512" argument se match karte hue).
    TARGET_SIZE = 512
    padded = padded.resize((TARGET_SIZE, TARGET_SIZE), Image.LANCZOS)

    padded.save(image_path)


def replace_background(input_video_path, bg_image_path, output_video_path):
    cap = cv2.VideoCapture(str(input_video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    bg = cv2.imread(str(bg_image_path))
    bg = cv2.resize(bg, (w, h))

    tmp_noaudio = str(output_video_path) + "_noaudio.mp4"
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(tmp_noaudio, fourcc, fps, (w, h))

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        rgba = remove(
            frame,
            session=rembg_session,
            alpha_matting=True,
            alpha_matting_foreground_threshold=240,
            alpha_matting_background_threshold=10,
            # ZAROORI: erode_size ko wapas 15 se 8 kiya — zyada erosion fine hair
            # strands (flyaway hairs) ko hi mask se erase kar deta hai, jisse edge
            # "smooth" to lagta hai lekin natural wispy hair ki jagah seedhi
            # lakeer jaisi dikhti hai. Kam erosion + zyada blur better balance hai.
            alpha_matting_erode_size=8,
        )
        alpha = rgba[:, :, 3:4].astype(np.float32) / 255.0
        # edges ko thoda soften karo taaki hard/patchy line na dikhe
        alpha = cv2.GaussianBlur(alpha, (31, 31), 0)
        # ZAROORI: cv2.GaussianBlur single-channel (H,W,1) input ko (H,W) bana deta hai —
        # channel dimension gayab ho jaati hai. Isi wajah se "operands could not be
        # broadcast together with shapes (H,W,3) (H,W)" error aata tha jab background
        # image upload ki jaati thi. Yahan wapas (H,W,1) shape restore kar rahe hain.
        if alpha.ndim == 2:
            alpha = alpha[:, :, np.newaxis]
        person = rgba[:, :, :3].astype(np.float32)
        composited = person * alpha + bg.astype(np.float32) * (1 - alpha)
        writer.write(np.clip(composited, 0, 255).astype(np.uint8))

    cap.release()
    writer.release()

    # ZAROORI: isi mux step ki wajah se pehle background-replace kaam nahi kar raha tha —
    # bina iske output_video_path kabhi banti hi nahi thi aur audio bhi miss ho jaata tha.
    mux_cmd = [
        "ffmpeg", "-y", "-i", tmp_noaudio, "-i", str(input_video_path),
        "-map", "0:v:0", "-map", "1:a:0?",
        "-c:v", "libx264", "-c:a", "aac", "-shortest",
        str(output_video_path)
    ]
    subprocess.run(mux_cmd, capture_output=True, text=True)


def overlay_foreground(input_video_path, fg_image_path, output_video_path, x_frac=0.7, y_frac=0.7, width_frac=0.3,
                        crop_top=0.0, crop_bottom=0.0, crop_left=0.0, crop_right=0.0):
    cap = cv2.VideoCapture(str(input_video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fg_img = cv2.imread(str(fg_image_path), cv2.IMREAD_UNCHANGED)

    # Foreground image ko user ke crop-top/bottom/left/right (0-0.9 fraction)
    # ke mutabik crop karo, resize karne se PEHLE.
    fh, fw = fg_img.shape[:2]
    top_px = int(fh * max(0.0, min(0.9, crop_top)))
    bottom_px = int(fh * max(0.0, min(0.9, crop_bottom)))
    left_px = int(fw * max(0.0, min(0.9, crop_left)))
    right_px = int(fw * max(0.0, min(0.9, crop_right)))
    y0, y1 = top_px, max(top_px + 1, fh - bottom_px)
    x0, x1 = left_px, max(left_px + 1, fw - right_px)
    fg_img = fg_img[y0:y1, x0:x1]

    fg_w = max(1, int(w * width_frac))
    ratio = fg_w / fg_img.shape[1]
    fg_h = max(1, int(fg_img.shape[0] * ratio))
    fg_resized = cv2.resize(fg_img, (fg_w, fg_h))

    if fg_resized.shape[2] == 4:
        fg_rgb = fg_resized[:, :, :3].astype(np.float32)
        fg_alpha = fg_resized[:, :, 3:4].astype(np.float32) / 255.0
    else:
        fg_rgb = fg_resized.astype(np.float32)
        fg_alpha = np.ones((fg_h, fg_w, 1), dtype=np.float32)

    pos_x = max(0, min(int(w * x_frac), w - fg_w))
    pos_y = max(0, min(int(h * y_frac), h - fg_h))

    tmp_noaudio = str(output_video_path) + "_noaudio.mp4"
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(tmp_noaudio, fourcc, fps, (w, h))

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        roi = frame[pos_y:pos_y + fg_h, pos_x:pos_x + fg_w].astype(np.float32)
        blended = fg_rgb * fg_alpha + roi * (1 - fg_alpha)
        frame[pos_y:pos_y + fg_h, pos_x:pos_x + fg_w] = blended.astype(np.uint8)
        writer.write(frame)

    cap.release()
    writer.release()

    mux_cmd = [
        "ffmpeg", "-y", "-i", tmp_noaudio, "-i", str(input_video_path),
        "-map", "0:v:0", "-map", "1:a:0?",
        "-c:v", "libx264", "-c:a", "aac", "-shortest",
        str(output_video_path)
    ]
    subprocess.run(mux_cmd, capture_output=True, text=True)


def apply_corner_light(input_video_path, color_hex, corner, output_video_path, base_intensity=0.25, motion_boost=0.5):
    color_hex = color_hex.lstrip("#")
    r, g, b = int(color_hex[0:2], 16), int(color_hex[2:4], 16), int(color_hex[4:6], 16)
    color_bgr = (b, g, r)

    cap = cv2.VideoCapture(str(input_video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    yy, xx = np.mgrid[0:h, 0:w]
    corner_map = {
        "top-left": (0, 0), "top-right": (w, 0),
        "bottom-left": (0, h), "bottom-right": (w, h),
    }
    cx, cy = corner_map.get(corner, (w, h))
    dist = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
    max_dist = np.sqrt(w ** 2 + h ** 2)
    gradient = np.clip(1 - (dist / max_dist), 0, 1) ** 2

    tmp_noaudio = str(output_video_path) + "_noaudio.mp4"
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(tmp_noaudio, fourcc, fps, (w, h))

    prev_gray = None
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        motion_level = 0.0
        if prev_gray is not None:
            diff = cv2.absdiff(gray, prev_gray)
            motion_level = float(np.mean(diff)) / 255.0
        prev_gray = gray

        intensity = base_intensity + motion_boost * min(motion_level * 5, 1.0)
        light_layer = np.zeros_like(frame, dtype=np.float32)
        for c in range(3):
            light_layer[:, :, c] = gradient * color_bgr[c] * intensity

        blended = np.clip(frame.astype(np.float32) + light_layer, 0, 255).astype(np.uint8)
        writer.write(blended)

    cap.release()
    writer.release()

    mux_cmd = [
        "ffmpeg", "-y", "-i", tmp_noaudio, "-i", str(input_video_path),
        "-map", "0:v:0", "-map", "1:a:0?",
        "-c:v", "libx264", "-c:a", "aac", "-shortest",
        str(output_video_path)
    ]
    subprocess.run(mux_cmd, capture_output=True, text=True)


BASE_DIR = Path("/content/SadTalker")
UPLOAD_DIR = Path("/content/uploads"); UPLOAD_DIR.mkdir(exist_ok=True)
RESULT_DIR = Path("/content/results"); RESULT_DIR.mkdir(exist_ok=True)

JOBS = {}  # job_id -> {"status": "processing"/"done"/"error", ...}


def run_job(job_id, image_path, script, voice_sample_path, background_path, light_color, light_corner, speech_speed):
    try:
        if voice_sample_path is not None:
            audio_path = UPLOAD_DIR / f"{job_id}.wav"
            clone_cmd = [
                "/content/xtts_env/bin/python", "/content/xtts_clone.py",
                script, str(voice_sample_path), str(audio_path)
            ]
            clone_env = os.environ.copy()
            clone_env["COQUI_TOS_AGREED"] = "1"
            clone_result = subprocess.run(clone_cmd, capture_output=True, text=True, env=clone_env)
            if clone_result.returncode != 0:
                JOBS[job_id] = {"status": "error", "error": "Voice cloning failed: " + clone_result.stderr[-500:]}
                return
        else:
            audio_path = UPLOAD_DIR / f"{job_id}.mp3"
            gTTS(text=script, lang="en", tld="co.in", slow=False).save(str(audio_path))

        # Speech speed adjust (atempo range 0.5-2.0 covers our 0.7-1.5 slider directly)
        speech_speed = max(0.7, min(1.5, float(speech_speed or 1.0)))
        if abs(speech_speed - 1.0) > 0.01:
            sped_path = UPLOAD_DIR / f"{job_id}_sped{audio_path.suffix}"
            atempo_cmd = [
                "ffmpeg", "-y", "-i", str(audio_path),
                "-filter:a", f"atempo={speech_speed}",
                str(sped_path)
            ]
            atempo_result = subprocess.run(atempo_cmd, capture_output=True, text=True)
            if atempo_result.returncode == 0 and sped_path.exists():
                audio_path = sped_path

        job_result_dir = RESULT_DIR / job_id
        job_result_dir.mkdir(exist_ok=True)

        JOBS[job_id] = {"status": "processing", "progress": 0, "started_at": time.time()}

        cmd = [
            "python3.8", "inference.py",
            "--driven_audio", str(audio_path.resolve()),
            "--source_image", str(image_path.resolve()),
            "--result_dir", str(job_result_dir.resolve()),
            "--preprocess", "full",
            "--size", "512",
            "--enhancer", "gfpgan",
        ]

        # Popen se real-time progress track karte hain (SadTalker apna tqdm progress
        # jaise "123/483" stderr pe likhta hai — usse parse karke asli % nikalte hain,
        # taaki frontend ko sach mein sahi ETA mil sake, guess-based nahi).
        proc = subprocess.Popen(cmd, cwd=str(BASE_DIR), stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT, text=True, bufsize=1)
        progress_pattern = re.compile(r"(\d+)/(\d+)")
        stderr_tail = []
        for line in proc.stdout:
            stderr_tail.append(line)
            if len(stderr_tail) > 60:
                stderr_tail.pop(0)
            match = progress_pattern.search(line)
            if match:
                done, total = int(match.group(1)), int(match.group(2))
                if total > 0:
                    pct = min(99, int(done / total * 100))
                    job = JOBS.get(job_id, {})
                    job["progress"] = pct
                    JOBS[job_id] = job
        proc.wait()

        if proc.returncode != 0:
            JOBS[job_id] = {"status": "error", "error": "".join(stderr_tail)[-500:]}
            return

        mp4_files = list(job_result_dir.glob("**/*.mp4"))
        if not mp4_files:
            JOBS[job_id] = {"status": "error", "error": "No output video found"}
            return

        current_video = mp4_files[0]
        job = JOBS.get(job_id, {})
        job.update({"status": "processing", "progress": 99})
        JOBS[job_id] = job

        if background_path is not None:
            bg_output = job_result_dir / f"{job_id}_with_bg.mp4"
            replace_background(current_video, background_path, bg_output)
            if bg_output.exists():
                current_video = bg_output

        if light_color and light_corner:
            light_output = job_result_dir / f"{job_id}_with_light.mp4"
            apply_corner_light(current_video, light_color, light_corner, light_output)
            if light_output.exists():
                current_video = light_output

        JOBS[job_id] = {"status": "done", "progress": 100, "video_path": str(current_video)}

    except Exception as e:
        import traceback
        JOBS[job_id] = {"status": "error", "error": f"{type(e).__name__}: {str(e)}", "trace": traceback.format_exc()[-1000:]}


FRONTEND_HTML = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>BolChehra — Talking Face Generator</title>
  <style>
    body { font-family: -apple-system, Segoe UI, Roboto, sans-serif; max-width: 560px; margin: 60px auto; padding: 0 20px; color: #1a1a1a; }
    .brand { display: flex; align-items: center; gap: 12px; margin-bottom: 4px; }
    .brand svg { width: 44px; height: 44px; flex-shrink: 0; }
    .brand h1 { font-size: 26px; margin: 0; letter-spacing: -0.5px; }
    p.sub { color: #666; margin-top: 4px; margin-bottom: 0; }
    label { display: block; font-weight: 600; margin-top: 20px; margin-bottom: 6px; }
    input[type="file"], textarea { width: 100%; padding: 10px; border: 1px solid #ccc; border-radius: 8px; font-size: 14px; box-sizing: border-box; }
    textarea { min-height: 90px; resize: vertical; }
    button { margin-top: 24px; width: 100%; padding: 12px; background: #d97757; color: white; border: none; border-radius: 8px; font-size: 15px; font-weight: 600; cursor: pointer; }
    button:disabled { background: #ccc; cursor: not-allowed; }
    #status { margin-top: 16px; font-size: 14px; color: #555; }
    video { margin-top: 20px; width: 100%; border-radius: 8px; }
    .limit-notice { margin-top: 8px; padding: 10px 12px; background: #fff4e5; border: 1px solid #f0b955; border-left: 4px solid #d97757; border-radius: 6px; font-size: 13px; color: #7a4a1a; line-height: 1.5; }
    .char-counter { margin-top: 6px; font-size: 12px; color: #888; text-align: right; }
    .char-counter.over { color: #c0392b; font-weight: 600; }
  </style>
</head>
<body>
  <div class="brand">
    <svg viewBox="0 0 100 100" xmlns="http://www.w3.org/2000/svg">
      <circle cx="50" cy="50" r="48" fill="#d97757" />
      <circle cx="50" cy="46" r="22" fill="#fff" />
      <circle cx="42" cy="42" r="3.2" fill="#d97757" />
      <circle cx="58" cy="42" r="3.2" fill="#d97757" />
      <path d="M40 54 Q50 62 60 54" stroke="#d97757" stroke-width="3" fill="none" stroke-linecap="round" />
      <path d="M14 38 Q8 50 14 62" stroke="#fff" stroke-width="4" fill="none" stroke-linecap="round" />
      <path d="M22 32 Q13 50 22 68" stroke="#fff" stroke-width="4" fill="none" stroke-linecap="round" opacity="0.6" />
      <path d="M86 38 Q92 50 86 62" stroke="#fff" stroke-width="4" fill="none" stroke-linecap="round" />
      <path d="M78 32 Q87 50 78 68" stroke="#fff" stroke-width="4" fill="none" stroke-linecap="round" opacity="0.6" />
    </svg>
    <h1>BolChehra</h1>
  </div>
  <p class="sub">Upload a photo, type a script, get a talking video.</p>

  <form id="genForm">
    <label for="image">Photo (face image)</label>
    <input type="file" id="image" accept="image/*" required>

    <label for="voiceSample">Apni awaaz ka sample (optional — 6-10 sec clean recording, video usi awaaz mein banega)</label>
    <input type="file" id="voiceSample" accept="audio/*">

    <label for="backgroundImage">Background image (optional — video ke peeche fit hoga)</label>
    <input type="file" id="backgroundImage" accept="image/*">

    <label for="foregroundImage">Foreground image (optional — video ke upar overlay hoga, PNG transparent best). Photo generate hone ke baad, asli video frame pe iski position set kar paoge.</label>
    <input type="file" id="foregroundImage" accept="image/*">

    <label style="display:flex;align-items:center;gap:8px;font-weight:600;margin-top:20px;">
      <input type="checkbox" id="enableLight" style="width:auto;">
      Corner light effect (movement ke saath react karega)
    </label>
    <div id="lightOptions" style="display:none;margin-top:10px;display:flex;gap:16px;">
      <div style="flex:1;">
        <label for="lightColor">Light color</label>
        <input type="color" id="lightColor" value="#00c8ff" style="height:40px;padding:2px;">
      </div>
      <div style="flex:1;">
        <label for="lightCorner">Corner</label>
        <select id="lightCorner" style="width:100%;padding:10px;border:1px solid #ccc;border-radius:8px;">
          <option value="top-left">Top-left</option>
          <option value="top-right">Top-right</option>
          <option value="bottom-left">Bottom-left</option>
          <option value="bottom-right" selected>Bottom-right</option>
        </select>
      </div>
    </div>

    <label for="speechSpeed">Awaaz ki speed (<span id="speechSpeedLabel">1.0x</span>)</label>
    <input type="range" id="speechSpeed" min="0.7" max="1.5" step="0.05" value="1.0" style="width:100%;">

    <label for="script">Script</label>
    <textarea id="script" placeholder="Apna script yahan likho..." required></textarea>
    <div class="char-counter" id="charCounter">0 / 5000 characters</div>
    <div class="limit-notice">
      ⚠️ <strong>Script 5000 characters se zyada na ho.</strong> Isse zyada lamba text ek saath process nahi ho payega — audio/video generation fail ho sakta hai. Lambe script ko chhote hisso mein todke alag-alag videos banao.
    </div>

    <button type="submit" id="submitBtn">Generate Video</button>
  </form>

  <div id="status"></div>
  <div id="timer" style="margin-top:6px;font-size:13px;color:#888;"></div>
  <div id="progressBarWrap" style="display:none;margin-top:10px;height:6px;background:#f0e5df;border-radius:4px;overflow:hidden;">
    <div id="progressBar" style="height:100%;width:0%;background:#d97757;transition:width 0.5s ease;"></div>
  </div>

  <div id="fgPositioner" style="display:none; margin-top:16px;">
    <p style="font-size:13px; color:#666; margin-bottom:6px;">Video ban gaya — ab foreground image ko asli video frame pe drag karke position set karo:</p>
    <div id="fgStage" style="position:relative; width:100%; max-width:400px; border:1px solid #ccc; border-radius:8px; overflow:hidden; background:#eee;">
      <img id="fgPreviewBase" style="width:100%; display:block; pointer-events:none;">
      <img id="fgPreviewOverlay" style="position:absolute; cursor:move; touch-action:none;">
    </div>
    <label style="margin-top:8px;">Size</label>
    <input type="range" id="fgSizeSlider" min="10" max="80" value="30" style="width:100%;">

    <p style="font-size:13px; color:#666; margin-top:14px; margin-bottom:4px;">Image ko zaroorat ke mutabik crop karo (top/bottom/left/right):</p>
    <div style="display:flex; gap:10px; flex-wrap:wrap;">
      <div style="flex:1; min-width:120px;">
        <label style="font-size:12px;">Top</label>
        <input type="range" id="fgCropTop" min="0" max="50" value="0" style="width:100%;">
      </div>
      <div style="flex:1; min-width:120px;">
        <label style="font-size:12px;">Bottom</label>
        <input type="range" id="fgCropBottom" min="0" max="50" value="0" style="width:100%;">
      </div>
      <div style="flex:1; min-width:120px;">
        <label style="font-size:12px;">Left</label>
        <input type="range" id="fgCropLeft" min="0" max="50" value="0" style="width:100%;">
      </div>
      <div style="flex:1; min-width:120px;">
        <label style="font-size:12px;">Right</label>
        <input type="range" id="fgCropRight" min="0" max="50" value="0" style="width:100%;">
      </div>
    </div>

    <button type="button" id="confirmFgBtn" style="margin-top:12px; width:100%; padding:12px; background:#d97757; color:white; border:none; border-radius:8px; font-size:15px; font-weight:600; cursor:pointer;">Mic position confirm karo aur final video banao</button>
  </div>

  <video id="resultVideo" controls style="display:none;"></video>

  <script>
    const API_URL = "";
    const form = document.getElementById("genForm");
    const statusEl = document.getElementById("status");
    const btn = document.getElementById("submitBtn");
    const videoEl = document.getElementById("resultVideo");
    const scriptEl = document.getElementById("script");
    const charCounterEl = document.getElementById("charCounter");
    const MAX_CHARS = 5000;
    const speechSpeedEl = document.getElementById("speechSpeed");
    const speechSpeedLabel = document.getElementById("speechSpeedLabel");

    speechSpeedEl.addEventListener("input", () => {
      speechSpeedLabel.textContent = parseFloat(speechSpeedEl.value).toFixed(2) + "x";
    });

    scriptEl.addEventListener("input", () => {
      const len = scriptEl.value.length;
      charCounterEl.textContent = `${len} / ${MAX_CHARS} characters`;
      charCounterEl.classList.toggle("over", len > MAX_CHARS);
    });

    document.getElementById("enableLight").addEventListener("change", (e) => {
      document.getElementById("lightOptions").style.display = e.target.checked ? "flex" : "none";
    });

    let fgXFrac = 0.7, fgYFrac = 0.7, fgWidthFrac = 0.3;
    let currentJobId = null;
    let pendingForegroundFile = null;
    let fgOriginalImg = null;  // asli (bina-crop) foreground image ka Image object
    const fgCanvas = document.createElement("canvas");  // crop draw karne ke liye, screen pe kabhi nahi dikhta
    const fgStage = document.getElementById("fgStage");
    const fgPreviewBase = document.getElementById("fgPreviewBase");
    const fgPreviewOverlay = document.getElementById("fgPreviewOverlay");
    const fgPositioner = document.getElementById("fgPositioner");
    const fgSizeSlider = document.getElementById("fgSizeSlider");
    const confirmFgBtn = document.getElementById("confirmFgBtn");
    const fgCropTop = document.getElementById("fgCropTop");
    const fgCropBottom = document.getElementById("fgCropBottom");
    const fgCropLeft = document.getElementById("fgCropLeft");
    const fgCropRight = document.getElementById("fgCropRight");

    function positionFgOverlayNow() {
      const stageW = fgStage.clientWidth;
      const stageH = fgStage.clientHeight;
      fgPreviewOverlay.style.left = (stageW * fgXFrac) + "px";
      fgPreviewOverlay.style.top = (stageH * fgYFrac) + "px";
    }

    function renderFgCrop() {
      // ZAROORI: pehle CSS clip-path use ho raha tha, jo image ko sirf VISUALLY
      // chhupata hai — uska bounding box (jitni jagah wo ghairta hai) same hi
      // rehta hai, isliye cropped hui khaali jagah drag ke liye available nahi
      // hoti thi. Ab hum canvas pe ASLI crop kar rahe hain — cropped image ka
      // box bhi utna hi chhota hota hai jitna dikh raha hai, isliye khaali hui
      // jagah mein mic ko move karna ab possible hai.
      if (!fgOriginalImg) return;
      const iw = fgOriginalImg.naturalWidth;
      const ih = fgOriginalImg.naturalHeight;
      const topPx = Math.round(ih * (fgCropTop.value / 100));
      const bottomPx = Math.round(ih * (fgCropBottom.value / 100));
      const leftPx = Math.round(iw * (fgCropLeft.value / 100));
      const rightPx = Math.round(iw * (fgCropRight.value / 100));
      const cw = Math.max(1, iw - leftPx - rightPx);
      const ch = Math.max(1, ih - topPx - bottomPx);

      fgCanvas.width = cw;
      fgCanvas.height = ch;
      const ctx = fgCanvas.getContext("2d");
      ctx.clearRect(0, 0, cw, ch);
      ctx.drawImage(fgOriginalImg, leftPx, topPx, cw, ch, 0, 0, cw, ch);
      fgPreviewOverlay.src = fgCanvas.toDataURL("image/png");

      // displayed width fgWidthFrac ke hisaab se, height cropped image ke
      // NAYE aspect ratio se — isse box bilkul cropped content jitna hi hota hai.
      const stageW = fgStage.clientWidth;
      const dispW = stageW * fgWidthFrac;
      const dispH = dispW * (ch / cw);
      fgPreviewOverlay.style.width = dispW + "px";
      fgPreviewOverlay.style.height = dispH + "px";
    }

    async function showFgPositionerWithRealFrame(jobId) {
      const frameRes = await fetch(`${API_URL}/frame/${jobId}`);
      const frameBlob = await frameRes.blob();
      fgPositioner.style.display = "block";

      // ZAROORI: pehle ek fixed 100ms wait karte the base image load hone ka —
      // agar image load hone mein zyada time lagta (slow connection), to
      // fgStage.clientWidth/Height galat (0 ya purani) value deta, jisse mic
      // ki initial position preview mein galat dikhti thi aur wahi galat
      // fraction final video mein bhi chala jaata tha. Ab hum image ke ASLI
      // "load" event ka wait karte hain, taaki measurement hamesha sahi ho —
      // chahe load hone mein jitna bhi time lage.
      await new Promise((resolve) => {
        fgPreviewBase.onload = resolve;
        fgPreviewBase.src = URL.createObjectURL(frameBlob);
      });

      fgOriginalImg = new Image();
      await new Promise((resolve) => {
        fgOriginalImg.onload = resolve;
        fgOriginalImg.src = URL.createObjectURL(pendingForegroundFile);
      });

      // ek extra animation-frame wait — taaki browser ne layout definitely
      // apply kar diya ho image load hone ke turant baad.
      await new Promise((resolve) => requestAnimationFrame(resolve));

      // Naye foreground image ke liye crop sliders reset karo, taaki purani
      // crop settings agli baar carry na ho.
      [fgCropTop, fgCropBottom, fgCropLeft, fgCropRight].forEach((s) => { s.value = 0; });

      renderFgCrop();
      positionFgOverlayNow();
    }

    fgSizeSlider.addEventListener("input", () => {
      fgWidthFrac = fgSizeSlider.value / 100;
      renderFgCrop();
    });

    [fgCropTop, fgCropBottom, fgCropLeft, fgCropRight].forEach((slider) => {
      slider.addEventListener("input", renderFgCrop);
    });

    let fgDragging = false, fgOffsetX = 0, fgOffsetY = 0;
    fgPreviewOverlay.addEventListener("pointerdown", (e) => {
      fgDragging = true;
      // ZAROORI: e.clientX/Y poori screen ke relative hain, jabki offsetLeft/Top
      // sirf fgStage ke relative — pehle inhe seedha subtract karne se drag ka
      // starting point galat calculate hota tha (mic click karte hi jump ho jaata
      // tha). getBoundingClientRect() se dono ko same (stage-relative) coordinate
      // system mein convert kar rahe hain.
      const stageRect = fgStage.getBoundingClientRect();
      fgOffsetX = (e.clientX - stageRect.left) - fgPreviewOverlay.offsetLeft;
      fgOffsetY = (e.clientY - stageRect.top) - fgPreviewOverlay.offsetTop;
      fgPreviewOverlay.setPointerCapture(e.pointerId);
    });
    fgPreviewOverlay.addEventListener("pointermove", (e) => {
      if (!fgDragging) return;
      const stageRect = fgStage.getBoundingClientRect();
      const stageW = fgStage.clientWidth;
      const stageH = fgStage.clientHeight;
      let newLeft = (e.clientX - stageRect.left) - fgOffsetX;
      let newTop = (e.clientY - stageRect.top) - fgOffsetY;
      newLeft = Math.max(0, Math.min(newLeft, stageW - fgPreviewOverlay.offsetWidth));
      newTop = Math.max(0, Math.min(newTop, stageH - fgPreviewOverlay.offsetHeight));
      fgPreviewOverlay.style.left = newLeft + "px";
      fgPreviewOverlay.style.top = newTop + "px";
      fgXFrac = newLeft / stageW;
      fgYFrac = newTop / stageH;
    });
    fgPreviewOverlay.addEventListener("pointerup", () => { fgDragging = false; });

    async function pollUntilDone(jobId, onProgress) {
      let failCount = 0;
      while (true) {
        await new Promise(r => setTimeout(r, 3000));
        try {
          const statusRes = await fetch(`${API_URL}/status/${jobId}`);
          if (!statusRes.ok) throw new Error("status check failed");
          const statusData = await statusRes.json();
          if (typeof statusData.progress === "number") onProgress(statusData.progress);
          if (statusData.status === "done") return;
          if (statusData.status === "error") throw new Error(statusData.error || "Something went wrong");
          failCount = 0;
        } catch (pollErr) {
          failCount++;
          if (failCount >= 5) throw new Error("Server se baar-baar connection fail ho raha hai: " + pollErr.message);
          await new Promise(r => setTimeout(r, 3000));
        }
      }
    }

    confirmFgBtn.addEventListener("click", async () => {
      confirmFgBtn.disabled = true;
      confirmFgBtn.textContent = "Mic add ho raha hai...";
      statusEl.textContent = "Foreground image final video mein add ho rahi hai...";
      try {
        const fgFormData = new FormData();
        fgFormData.append("foreground_image", pendingForegroundFile);
        fgFormData.append("fg_x", fgXFrac);
        fgFormData.append("fg_y", fgYFrac);
        fgFormData.append("fg_width", fgWidthFrac);
        fgFormData.append("crop_top", fgCropTop.value / 100);
        fgFormData.append("crop_bottom", fgCropBottom.value / 100);
        fgFormData.append("crop_left", fgCropLeft.value / 100);
        fgFormData.append("crop_right", fgCropRight.value / 100);
        const applyRes = await fetch(`${API_URL}/apply-foreground/${currentJobId}`, { method: "POST", body: fgFormData });
        if (!applyRes.ok) throw new Error("Foreground apply karne mein error aayi");

        await pollUntilDone(currentJobId, () => {});

        const videoRes = await fetch(`${API_URL}/video/${currentJobId}`);
        const blob = await videoRes.blob();
        videoEl.src = URL.createObjectURL(blob);
        videoEl.style.display = "block";
        statusEl.textContent = "Done!";
        fgPositioner.style.display = "none";
      } catch (err) {
        statusEl.textContent = "Error: " + err.message;
      } finally {
        confirmFgBtn.disabled = false;
        confirmFgBtn.textContent = "Mic position confirm karo aur final video banao";
      }
    });

    form.addEventListener("submit", async (e) => {
      e.preventDefault();
      const imageFile = document.getElementById("image").files[0];
      const script = document.getElementById("script").value;
      if (!imageFile || !script) return;
      if (script.length > MAX_CHARS) {
        statusEl.textContent = `Script bahut lamba hai (${script.length} characters).`;
        return;
      }
      const formData = new FormData();
      formData.append("image", imageFile);
      formData.append("script", script);
      const voiceFile = document.getElementById("voiceSample").files[0];
      if (voiceFile) formData.append("voice_sample", voiceFile);
      const bgFile = document.getElementById("backgroundImage").files[0];
      if (bgFile) formData.append("background_image", bgFile);
      pendingForegroundFile = document.getElementById("foregroundImage").files[0] || null;
      if (document.getElementById("enableLight").checked) {
        formData.append("light_color", document.getElementById("lightColor").value);
        formData.append("light_corner", document.getElementById("lightCorner").value);
      }
      formData.append("speech_speed", speechSpeedEl.value);

      btn.disabled = true;
      btn.textContent = "Generating... (this can take a few minutes)";
      statusEl.textContent = "Processing your video, please wait...";
      videoEl.style.display = "none";
      fgPositioner.style.display = "none";
      document.getElementById("progressBarWrap").style.display = "block";

      let elapsedSec = 0;
      let lastProgress = 0;
      const timerEl = document.getElementById("timer");
      const progressBarEl = document.getElementById("progressBar");
      const timerMessage = (m, s) => {
        let etaText = "calculating...";
        if (lastProgress > 3 && elapsedSec > 5) {
          const totalEstSec = elapsedSec / (lastProgress / 100);
          const remainingSec = Math.max(0, Math.round(totalEstSec - elapsedSec));
          const rm = Math.floor(remainingSec / 60);
          const rs = (remainingSec % 60).toString().padStart(2, "0");
          etaText = `~${rm}:${rs} baaki`;
        }
        return `Time elapsed: ${m}:${s} — ${lastProgress}% complete — ${etaText}`;
      };
      timerEl.textContent = timerMessage("0", "00");
      const timerInterval = setInterval(() => {
        elapsedSec++;
        const m = Math.floor(elapsedSec / 60);
        const s = (elapsedSec % 60).toString().padStart(2, "0");
        timerEl.textContent = timerMessage(m, s);
      }, 1000);

      try {
        const res = await fetch(`${API_URL}/generate`, { method: "POST", body: formData });
        const data = await res.json();
        if (!res.ok) throw new Error(data.error || "Something went wrong");
        currentJobId = data.job_id;

        await pollUntilDone(currentJobId, (p) => {
          lastProgress = p;
          progressBarEl.style.width = p + "%";
        });

        if (pendingForegroundFile) {
          statusEl.textContent = "Base video ban gaya — ab mic ki position set karo neeche.";
          await showFgPositionerWithRealFrame(currentJobId);
        } else {
          const videoRes = await fetch(`${API_URL}/video/${currentJobId}`);
          if (!videoRes.ok) throw new Error("video not ready yet");
          const blob = await videoRes.blob();
          videoEl.src = URL.createObjectURL(blob);
          videoEl.style.display = "block";
          statusEl.textContent = "Done!";
        }
      } catch (err) {
        statusEl.textContent = "Error: " + err.message;
      } finally {
        clearInterval(timerInterval);
        document.getElementById("progressBarWrap").style.display = "none";
        btn.disabled = false;
        btn.textContent = "Generate Video";
      }
    });
  </script>
</body>
</html>
"""

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])


@app.get("/", response_class=HTMLResponse)
def serve_frontend():
    return FRONTEND_HTML


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/generate")
async def generate(
    image: UploadFile = File(...),
    script: str = Form(...),
    voice_sample: UploadFile = File(None),
    background_image: UploadFile = File(None),
    light_color: str = Form(None),
    light_corner: str = Form(None),
    speech_speed: float = Form(1.0),
):
    job_id = str(uuid.uuid4())[:8]

    image_path = UPLOAD_DIR / f"{job_id}_{image.filename}"
    with open(image_path, "wb") as f:
        shutil.copyfileobj(image.file, f)
    center_crop_square(image_path)

    voice_sample_path = None
    if voice_sample is not None:
        voice_sample_path = UPLOAD_DIR / f"{job_id}_sample.wav"
        with open(voice_sample_path, "wb") as f:
            shutil.copyfileobj(voice_sample.file, f)

    background_path = None
    if background_image is not None:
        background_path = UPLOAD_DIR / f"{job_id}_bg.png"
        with open(background_path, "wb") as f:
            shutil.copyfileobj(background_image.file, f)

    JOBS[job_id] = {"status": "processing", "progress": 0}
    thread = threading.Thread(
        target=run_job,
        args=(job_id, image_path, script, voice_sample_path, background_path, light_color, light_corner, speech_speed)
    )
    thread.start()

    return JSONResponse({"job_id": job_id})


@app.get("/status/{job_id}")
def status(job_id: str):
    job = JOBS.get(job_id)
    if job is None:
        return JSONResponse(status_code=404, content={"error": "Job not found"})
    return job


@app.get("/video/{job_id}")
def video(job_id: str):
    job = JOBS.get(job_id)
    if job is None or job.get("status") != "done":
        return JSONResponse(status_code=404, content={"error": "Video not ready"})
    return FileResponse(job["video_path"], media_type="video/mp4", filename=f"{job_id}.mp4")


@app.get("/frame/{job_id}")
def frame(job_id: str):
    """Base video (jo abhi 'done' hai) ka ASLI pehla frame nikalta hai — isi ke
    upar user mic/foreground ki position set karega. Isse guarantee hota hai ki
    preview mein jo dikhega, wahi final video mein hoga — kyunki yeh koi alag
    photo nahi, wahi actual video frame hai."""
    job = JOBS.get(job_id)
    if job is None or job.get("status") != "done":
        return JSONResponse(status_code=404, content={"error": "Video not ready"})
    cap = cv2.VideoCapture(job["video_path"])
    ret, frame_img = cap.read()
    cap.release()
    if not ret:
        return JSONResponse(status_code=500, content={"error": "Could not read frame"})
    frame_path = UPLOAD_DIR / f"{job_id}_frame.png"
    cv2.imwrite(str(frame_path), frame_img)
    return FileResponse(frame_path, media_type="image/png")


def apply_fg_job(job_id, base_video_path, foreground_path, fg_x, fg_y, fg_width,
                  crop_top, crop_bottom, crop_left, crop_right):
    try:
        job_result_dir = RESULT_DIR / job_id
        job_result_dir.mkdir(exist_ok=True)
        fg_output = job_result_dir / f"{job_id}_final_with_fg.mp4"
        overlay_foreground(base_video_path, foreground_path, fg_output, x_frac=fg_x, y_frac=fg_y, width_frac=fg_width,
                            crop_top=crop_top, crop_bottom=crop_bottom, crop_left=crop_left, crop_right=crop_right)
        JOBS[job_id] = {"status": "done", "progress": 100, "video_path": str(fg_output)}
    except Exception as e:
        import traceback
        JOBS[job_id] = {"status": "error", "error": f"{type(e).__name__}: {str(e)}", "trace": traceback.format_exc()[-1000:]}


@app.post("/apply-foreground/{job_id}")
async def apply_foreground(
    job_id: str,
    foreground_image: UploadFile = File(...),
    fg_x: float = Form(0.7),
    fg_y: float = Form(0.7),
    fg_width: float = Form(0.3),
    crop_top: float = Form(0.0),
    crop_bottom: float = Form(0.0),
    crop_left: float = Form(0.0),
    crop_right: float = Form(0.0),
):
    job = JOBS.get(job_id)
    if job is None or job.get("status") != "done":
        return JSONResponse(status_code=404, content={"error": "Base video abhi ready nahi hai"})

    base_video_path = job["video_path"]
    foreground_path = UPLOAD_DIR / f"{job_id}_fg.png"
    with open(foreground_path, "wb") as f:
        shutil.copyfileobj(foreground_image.file, f)

    JOBS[job_id] = {"status": "processing", "progress": 50}
    thread = threading.Thread(
        target=apply_fg_job,
        args=(job_id, base_video_path, foreground_path, fg_x, fg_y, fg_width,
              crop_top, crop_bottom, crop_left, crop_right)
    )
    thread.start()

    return JSONResponse({"status": "processing"})


public_url = ngrok.connect(8000)
print("Tumhara public API link:", public_url)

nest_asyncio.apply()
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
loop = asyncio.get_event_loop()
loop.run_until_complete(server.serve())